# File 07 LR — Lighting-Robust Grad-CAM
Explainability for lighting-robust classifier. No training.


In [1]:
import os, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.models import efficientnet_b0
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    USE_LIB=True
except ImportError:
    USE_LIB=False; print('pytorch_grad_cam unavailable, manual fallback')

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MANIFEST_PATH=Path(r'D:\DIABETES\diabetes_pipeline_outputs\03_segmented_export_manifest.csv')
PRED_PATH=Path(r'D:\DIABETES\diabetes_pipeline_outputs\06_segmented_test_evaluation_lighting_robust\06_lr_test_predictions.csv')
CHECKPOINT=Path(r'D:\DIABETES\diabetes_pipeline_outputs\05_segmented_training_lighting_robust\best_segmented_lighting_robust_model.pth')
SUMMARY_JSON=Path(r'D:\DIABETES\diabetes_pipeline_outputs\05_segmented_training_lighting_robust\05_lr_best_checkpoint_summary.json')
OUTPUT_DIR=Path(r'D:\DIABETES\diabetes_pipeline_outputs\07_segmented_gradcam_lighting_robust')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for d in ['false_positives','false_negatives','true_positives_high_confidence','true_negatives_high_confidence','borderline_cases','suspicious_segmentation_cases','non_suspicious_cases','edge_touch_cases','non_edge_touch_cases','random_balanced_cases','contact_sheets']:
    (OUTPUT_DIR/d).mkdir(exist_ok=True)
IMG_SIZE=224; MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
print(f'Device: {DEVICE}')


Device: cuda


In [2]:
def correct_lighting_clahe(image_pil, clip_limit=1.5, tile_grid_size=(8,8)):
    img_rgb=np.array(image_pil.convert('RGB'))
    lab=cv2.cvtColor(img_rgb,cv2.COLOR_RGB2LAB)
    l,a,b=cv2.split(lab)
    clahe=cv2.createCLAHE(clipLimit=clip_limit,tileGridSize=tile_grid_size)
    l_eq=clahe.apply(l)
    return Image.fromarray(cv2.cvtColor(cv2.merge([l_eq,a,b]),cv2.COLOR_LAB2RGB))

def pad_square(img):
    w,h=img.size; s=max(w,h); new=Image.new('RGB',(s,s),(0,0,0)); new.paste(img,((s-w)//2,(s-h)//2)); return new

transform=T.Compose([T.Lambda(correct_lighting_clahe),T.Lambda(pad_square),T.Resize((IMG_SIZE,IMG_SIZE)),T.ToTensor(),T.Normalize(mean=MEAN,std=STD)])

df_manifest=pd.read_csv(MANIFEST_PATH)
df_pred=pd.read_csv(PRED_PATH)
if SUMMARY_JSON.exists():
    with open(SUMMARY_JSON) as f: SELECTED_THRESHOLD=float(json.load(f).get('best_threshold',0.5))
else:
    SELECTED_THRESHOLD=0.5; print('WARNING: fallback threshold')

if 'image_path' in df_pred.columns:
    qc_cols=[c for c in ['suspicious_large_mask','suspicious_edge_touch','mask_foreground_ratio','bbox_area_ratio','segmentation_status'] if c in df_manifest.columns]
    if 'segmented_image_path' in df_manifest.columns:
        df_pred=df_pred.merge(df_manifest[['segmented_image_path']+qc_cols].rename(columns={'segmented_image_path':'image_path'}),on='image_path',how='left')

model=efficientnet_b0(weights=None)
model.classifier=nn.Sequential(nn.Dropout(0.3),nn.Linear(model.classifier[1].in_features,1))
model.load_state_dict(torch.load(CHECKPOINT,map_location=DEVICE))
model=model.to(DEVICE); model.eval()
TARGET_LAYER=model.features[-1]
print(f'Model loaded. Target: {TARGET_LAYER.__class__.__name__} Threshold:{SELECTED_THRESHOLD:.3f}')


Model loaded. Target: Conv2dNormActivation Threshold:0.500


In [3]:
# =========================
# Grad-CAM library availability
# =========================

try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    USE_GRADCAM_LIB = True
    print("Using pytorch-grad-cam library.")
except Exception as e:
    USE_GRADCAM_LIB = False
    print("pytorch-grad-cam unavailable. Using manual Grad-CAM fallback.")
    print(e)

Using pytorch-grad-cam library.


In [4]:
# =========================
# Grad-CAM Functions - Robust Binary Output Fix
# =========================

def pad_square(img):
    w, h = img.size
    s = max(w, h)
    new = Image.new("RGB", (s, s), (0, 0, 0))
    new.paste(img, ((s - w) // 2, (s - h) // 2))
    return new


transform = T.Compose([
    T.Lambda(correct_lighting_clahe),   # lighting robust version
    T.Lambda(pad_square),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])


if USE_GRADCAM_LIB:
    cam = GradCAM(model=model, target_layers=[TARGET_LAYER])

    class BinaryClassifierTarget:
        def __init__(self, positive=True):
            self.positive = positive

        def __call__(self, output):
            # Robust for scalar, [1], [B], or [B, 1]
            score = output.reshape(-1)[0]
            return score if self.positive else -score

    def generate_gradcam(img_path, target_diabetes=True):
        img_pil = Image.open(img_path).convert("RGB")

        # Display version should match the model input pipeline
        img_display = correct_lighting_clahe(img_pil)
        img_display = pad_square(img_display)
        img_display = img_display.resize((IMG_SIZE, IMG_SIZE))

        img_rgb = np.array(img_display).astype(np.float32) / 255.0

        inp = transform(img_pil).unsqueeze(0).to(DEVICE)

        targets = [BinaryClassifierTarget(positive=target_diabetes)]
        heatmap = cam(input_tensor=inp, targets=targets)[0]

        overlay = show_cam_on_image(img_rgb, heatmap, use_rgb=True)

        return img_rgb, heatmap, overlay


else:
    def generate_gradcam(img_path, target_diabetes=True):
        model.eval()

        activations = {}
        gradients = {}

        def forward_hook(module, input, output):
            activations["value"] = output

        def backward_hook(module, grad_input, grad_output):
            gradients["value"] = grad_output[0]

        forward_handle = TARGET_LAYER.register_forward_hook(forward_hook)
        backward_handle = TARGET_LAYER.register_full_backward_hook(backward_hook)

        try:
            img_pil = Image.open(img_path).convert("RGB")

            # Display version should match model input pipeline
            img_display = correct_lighting_clahe(img_pil)
            img_display = pad_square(img_display)
            img_display = img_display.resize((IMG_SIZE, IMG_SIZE))

            img_rgb = np.array(img_display).astype(np.float32) / 255.0

            inp = transform(img_pil).unsqueeze(0).to(DEVICE)

            model.zero_grad(set_to_none=True)

            out = model(inp)

            # Robust for scalar, [1], [B], or [B, 1]
            logit = out.reshape(-1)[0]

            target = logit if target_diabetes else -logit
            target.backward()

            if "value" not in activations or "value" not in gradients:
                raise RuntimeError("Grad-CAM hooks failed: missing activations or gradients.")

            acts = activations["value"].detach()
            grads = gradients["value"].detach()

            if acts.ndim != 4 or grads.ndim != 4:
                raise RuntimeError(f"Unexpected Grad-CAM shapes: acts={acts.shape}, grads={grads.shape}")

            weights = grads.mean(dim=(2, 3), keepdim=True)
            cam_map = (weights * acts).sum(dim=1, keepdim=True)
            cam_map = torch.relu(cam_map)

            heatmap = cam_map.squeeze().detach().cpu().numpy()

            if heatmap.max() > heatmap.min():
                heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min())
            else:
                heatmap = np.zeros_like(heatmap)

            heatmap = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

            heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
            heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

            overlay = (0.55 * img_rgb + 0.45 * heatmap_colored)
            overlay = np.clip(overlay, 0, 1)

            return img_rgb, heatmap, overlay

        finally:
            forward_handle.remove()
            backward_handle.remove()


print("Grad-CAM functions ready.")

Grad-CAM functions ready.


In [5]:
df_pred['correct']=(df_pred['y_pred_sel']==df_pred['y_true'])
df_pred['fp']=(df_pred['y_pred_sel']==1)&(df_pred['y_true']==0)
df_pred['fn']=(df_pred['y_pred_sel']==0)&(df_pred['y_true']==1)
df_pred['tp']=(df_pred['y_pred_sel']==1)&(df_pred['y_true']==1)
df_pred['tn']=(df_pred['y_pred_sel']==0)&(df_pred['y_true']==0)
df_pred['dist']=np.abs(df_pred['y_prob_diabetes']-SELECTED_THRESHOLD)

cases={}
cases['false_positives']=df_pred[df_pred['fp']].copy()
cases['false_negatives']=df_pred[df_pred['fn']].copy()
cases['true_positives_high_confidence']=df_pred[df_pred['tp']].nlargest(12,'y_prob_diabetes')
cases['true_negatives_high_confidence']=df_pred[df_pred['tn']].nsmallest(12,'y_prob_diabetes')
cases['borderline_cases']=df_pred.nsmallest(12,'dist')
cases['random_balanced_cases']=pd.concat([df_pred[df_pred['y_true']==1].sample(min(12,(df_pred['y_true']==1).sum()),random_state=SEED),df_pred[df_pred['y_true']==0].sample(min(12,(df_pred['y_true']==0).sum()),random_state=SEED)])
for col,flag,key in [('suspicious_edge_touch',True,'edge_touch_cases'),('suspicious_edge_touch',False,'non_edge_touch_cases')]:
    if col in df_pred.columns:
        sub=df_pred[df_pred[col]==flag]; cases[key]=sub.sample(min(12,len(sub)),random_state=SEED)
if 'segmentation_status' in df_pred.columns:
    ok=df_pred[df_pred['segmentation_status']=='ok']; cases['non_suspicious_cases']=ok.sample(min(12,len(ok)),random_state=SEED)
    susp=df_pred[df_pred['segmentation_status']!='ok']; cases['suspicious_segmentation_cases']=susp.sample(min(12,len(susp)),random_state=SEED)

pd.DataFrame([{'group':k,'count':len(v)} for k,v in cases.items()]).to_csv(OUTPUT_DIR/'07_lr_gradcam_case_selection_summary.csv',index=False)
print('Cases selected.')


Cases selected.


In [6]:
review=[]
for gname,gdf in cases.items():
    for i,(idx,row) in enumerate(gdf.iterrows()):
        try:
            ir,hm,ov=generate_gradcam(row['image_path'],target_diabetes=True)
            cid=f'{gname}_{i}'
            fig,axes=plt.subplots(1,3,figsize=(15,5))
            axes[0].imshow(ir); axes[0].set_title('Input(CLAHE)'); axes[0].axis('off')
            axes[1].imshow(hm,cmap='jet'); axes[1].set_title('Grad-CAM'); axes[1].axis('off')
            axes[2].imshow(ov); axes[2].set_title('Overlay'); axes[2].axis('off')
            plt.suptitle(f'True:{row["final_label"]} Prob:{row["y_prob_diabetes"]:.3f}',fontsize=9)
            plt.tight_layout()
            pp=OUTPUT_DIR/gname/f'{cid}.png'; plt.savefig(pp,dpi=100); plt.close()
            com=ndimage.center_of_mass(hm) if hm.sum()>0 else (0,0)
            em=np.zeros_like(hm); em[0,:]=em[-1,:]=em[:,0]=em[:,-1]=1
            ea=(hm*em).sum()/max(hm.sum(),1e-9)
            s=max(IMG_SIZE//4,1); cm2=np.zeros_like(hm); cm2[:s,:s]=cm2[:s,-s:]=cm2[-s:,:s]=cm2[-s:,-s:]=1
            ca=(hm*cm2).sum()/max(hm.sum(),1e-9)
            review.append({'case_id':cid,'segmented_image_path':row['image_path'],'final_label':row.get('final_label',''),'label_binary':row.get('y_true',None),'probability_diabetes':row['y_prob_diabetes'],'prediction_at_selected_threshold':row.get('y_pred_sel',None),'prediction_correct':row.get('correct',None),'case_group':gname,'segmentation_status':row.get('segmentation_status',''),'suspicious_large_mask':row.get('suspicious_large_mask',''),'suspicious_edge_touch':row.get('suspicious_edge_touch',''),'mask_foreground_ratio':row.get('mask_foreground_ratio',''),'bbox_area_ratio':row.get('bbox_area_ratio',''),'gradcam_target_used':'diabetes_positive_logit','gradcam_file_path':str(pp),'heatmap_center_of_mass_x':float(com[1]),'heatmap_center_of_mass_y':float(com[0]),'heatmap_edge_attention_ratio':float(ea),'heatmap_corner_attention_ratio':float(ca),'heatmap_max_value':float(hm.max()),'heatmap_mean_value':float(hm.mean())})
        except Exception as e:
            print(f'Failed {gname}_{i}: {e}')
pd.DataFrame(review).to_csv(OUTPUT_DIR/'07_lr_gradcam_review_manifest.csv',index=False)
print(f'Grad-CAM images: {len(review)}')


Grad-CAM images: 100


In [7]:
df_rev=pd.DataFrame(review)
df_rev[['case_id','heatmap_center_of_mass_x','heatmap_center_of_mass_y','heatmap_edge_attention_ratio','heatmap_corner_attention_ratio','heatmap_max_value','heatmap_mean_value']].to_csv(OUTPUT_DIR/'07_lr_gradcam_heatmap_diagnostics.csv',index=False)
shortcut=df_rev[(df_rev['heatmap_edge_attention_ratio']>0.3)|(df_rev['heatmap_corner_attention_ratio']>0.25)]
shortcut.to_csv(OUTPUT_DIR/'07_lr_shortcut_risk_cases.csv',index=False)
print(f'Shortcut risk: {len(shortcut)}')

for gname in cases.keys():
    files=list((OUTPUT_DIR/gname).glob('*.png'))
    if not files: continue
    n=min(len(files),12); ncols=3; nrows=(n+2)//ncols
    fig,axes=plt.subplots(nrows,ncols,figsize=(15,nrows*5))
    axes=np.array(axes).flatten() if nrows>1 or ncols>1 else [axes]
    for i,fp in enumerate(files[:n]):
        axes[i].imshow(plt.imread(fp)); axes[i].axis('off')
    for i in range(n,len(axes)): axes[i].axis('off')
    plt.suptitle(gname,fontsize=12); plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'contact_sheets'/f'07_lr_contact_{gname}.png',dpi=100); plt.close()

# Compare to old File 07
old_sc=Path(r'D:\DIABETES\diabetes_pipeline_outputs\07_segmented_gradcam_explainability\07_segmented_shortcut_risk_cases.csv')
comp_str=f'\nOld shortcut risk:{len(pd.read_csv(old_sc))} New:{len(shortcut)}' if old_sc.exists() else ''
json.dump({'checkpoint':str(CHECKPOINT),'threshold':SELECTED_THRESHOLD,'gradcam_method':'pytorch_grad_cam' if USE_LIB else 'manual','target_layer':str(TARGET_LAYER.__class__.__name__)},open(OUTPUT_DIR/'07_lr_gradcam_config.json','w'),indent=2)
handoff=f'FILE 07 LR HANDOFF\nStatus: PASS\nCheckpoint: {CHECKPOINT}\nGrad-CAM: {"pytorch_grad_cam" if USE_LIB else "manual"}\nTarget: {TARGET_LAYER.__class__.__name__}\nThreshold: {SELECTED_THRESHOLD:.3f}\nImages generated: {len(review)}\nShortcut risk: {len(shortcut)}{comp_str}\nGrad-CAM is qualitative. Does not prove causality.\n'
with open(OUTPUT_DIR/'07_lr_gradcam_handoff_summary.txt','w') as f: f.write(handoff)
print(handoff)


Shortcut risk: 25
FILE 07 LR HANDOFF
Status: PASS
Checkpoint: D:\DIABETES\diabetes_pipeline_outputs\05_segmented_training_lighting_robust\best_segmented_lighting_robust_model.pth
Grad-CAM: pytorch_grad_cam
Target: Conv2dNormActivation
Threshold: 0.500
Images generated: 100
Shortcut risk: 25
Grad-CAM is qualitative. Does not prove causality.

